In [1]:
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

from datetime import timedelta
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans


import itertools

In [2]:
directory = '../datasets/year_month=22-09/plugin=slurm_pub/metric=s21.cluster_cpu_util/a_0.parquet'
cpu_util_df = pd.read_parquet(directory)
mem_util_dir = '../datasets/year_month=22-09/plugin=slurm_pub/metric=s21.cluster_mem_util/a_0.parquet'
mem_util_df = pd.read_parquet(mem_util_dir)
directory = '../datasets/year_month=22-09/plugin=slurm_pub/metric=s21.cluster_gpu_util/a_0.parquet'
gpu_util_df = pd.read_parquet(directory)
jobid_dir = '../datasets/year_month=22-09/plugin=slurm_pub/metric=job_id/a_0.parquet'
job_id_df = pd.read_parquet(jobid_dir)
jobtable_dir = '../datasets/year_month=22-09/plugin=job_table/metric=job_info_marconi100/a_0.parquet'
job_table_df = pd.read_parquet(jobtable_dir)

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

#PLOTTING 1 RESOURCE AT A TIME
def plot_res_util_pct_with_waste_plotly(user_id, job_id, util_df, resource):

    # -------------------------
    # Filter job metadata
    # -------------------------
    user_df_1 = job_table_df[job_table_df['user_id'] == user_id]
    user_df_2 = job_id_df[job_id_df['user_id'] == user_id]

    start_time = pd.to_datetime(
        user_df_1[user_df_1['job_id'] == job_id]['start_time'].iloc[0]
    )
    end_time = pd.to_datetime(
        user_df_1[user_df_1['job_id'] == job_id]['end_time'].iloc[0]
    )

    partition_num = user_df_2[
        user_df_2['value'] == job_id
    ]['partition'].iloc[0]

    # -------------------------
    # Utilization slice
    # -------------------------
    x_df = util_df[
        (util_df['timestamp'] >= start_time) &
        (util_df['timestamp'] <= end_time) &
        (util_df['partition'] == partition_num)
    ].copy()

    if x_df.empty:
        print("No utilization data found.")
        return None

    x_df["timestamp"] = pd.to_datetime(x_df["timestamp"])
    x_df = x_df.sort_values("timestamp")

    # Clamp values to [0, 100]
    x_df["value"] = x_df["value"].astype(float).clip(lower=0, upper=100)

    # -------------------------
    # Plotly Figure
    # -------------------------
    fig = go.Figure()

    # Utilization line
    fig.add_trace(go.Scatter(
        x=x_df["timestamp"],
        y=x_df["value"],
        mode="lines",
        name=f"{resource} utilized (%)",
        hovertemplate="Time: %{x}<br>Util: %{y:.2f}%<extra></extra>"
    ))

    # Allocation baseline (100%)
    fig.add_trace(go.Scatter(
        x=x_df["timestamp"],
        y=np.full(len(x_df), 100.0),
        mode="lines",
        line=dict(width=0),
        name="Allocation (100%)",
        hoverinfo="skip",
        showlegend=False
    ))

    # Idle capacity shaded region
    fig.add_trace(go.Scatter(
        x=x_df["timestamp"],
        y=x_df["value"],
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        name="Idle capacity gap",
        hovertemplate="Time: %{x}<br>Idle gap: %{customdata:.2f}%<extra></extra>",
        customdata=(100.0 - x_df["value"]).to_numpy()
    ))

    # 100% reference line
    fig.add_hline(y=100, line_dash="dash")

    fig.update_layout(
        title=f"{resource} Utilization (user={user_id}, job={job_id}, partition={partition_num})",
        xaxis=dict(
            title="Time",
            rangeslider=dict(visible=True),
            type="date"
        ),
        yaxis=dict(
            title="Utilization (%)",
            range=[0, 105]
        ),
        hovermode="x unified",
        legend=dict(orientation="h"),
        template="plotly_white",
        margin=dict(l=60, r=20, t=120, b=60)
    )

    fig.show()

    return partition_num, start_time, end_time


def run_plot_interactive_plotly():

    user_id = int(input("Enter a user ID: "))
    job_id = int(input("Enter a job ID: "))
    resource = input("MEMORY / CPU / GPU: ").strip().upper()

    valid1 = (
        (job_table_df["user_id"] == user_id) &
        (job_table_df["job_id"] == job_id)
    ).any()

    valid2 = (
        (job_id_df["user_id"] == user_id) &
        (job_id_df["value"] == job_id)
    ).any()

    if not valid1 or not valid2:
        print("Invalid user/job combination.")
        return None

    if resource == "MEMORY":
        util_df = mem_util_df
    elif resource == "CPU":
        util_df = cpu_util_df
    elif resource == "GPU":
        util_df = gpu_util_df
    else:
        print("Invalid resource.")
        return None

    out = plot_res_util_pct_with_waste_plotly(
        user_id,
        job_id,
        util_df,
        resource
    )

    if out is None:
        return None

    part, start, end = out

    print(
        f"Plotted: user={user_id}, job={job_id}, "
        f"resource={resource}, partition={part}, "
        f"window={start} → {end}"
    )

    return out

In [11]:
run_plot_interactive_plotly()

Enter a user ID: 2
Enter a job ID: 1582167
MEMORY / CPU / GPU: GPU


Plotted: user=2, job=1582167, resource=GPU, partition=1, window=2022-09-15 06:11:39+00:00 → 2022-09-15 14:31:41+00:00


('1',
 Timestamp('2022-09-15 06:11:39+0000', tz='UTC'),
 Timestamp('2022-09-15 14:31:41+0000', tz='UTC'))